# Capítulo 5 · Actividad 05: selección de características en enfermedad cardiaca

> ¿Necesito las 12 columnas del proyecto 1 para detectar enfermedad cardiaca, o con menos queda igual de bien?

En la clase 09 se aplicaron varias técnicas de selección de características al dataset Wine. Aquí las aplico a los datos de enfermedad cardiaca del proyecto 1 del capítulo 5 (`01_proyecto_comparacion_modelos.ipynb`) y mantengo sus decisiones:

- La misma tabla, `data/heart_disease_uci.csv`, con las mismas reglas de limpieza: `chol` y `trestbps` en 0 pasan a faltantes, y `target = 1` si `num` es 1, 2, 3 o 4.
- Las mismas 12 columnas candidatas: 5 numéricas (`age`, `trestbps`, `chol`, `thalch`, `oldpeak`) y 7 categóricas (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `thal`). `ca` y `dataset` siguen fuera por las decisiones 6 y 7 del proyecto.
- La misma partición 60/20/20 para entrenamiento, validación y prueba, con `stratify` y `random_state=42`, y los mismos tres modelos: regresión logística, KNN (vecinos más cercanos) y árbol de decisión.

Seleccionar características es quedarse con algunas de las columnas originales y quitar las demás. Ajusto todas las técnicas solo con entrenamiento, uso validación para elegir y uso la prueba una sola vez, al final.

In [ ]:
import time
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import RFE, SequentialFeatureSelector, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, fbeta_score, make_scorer, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Que scikit-learn devuelva DataFrames (con nombres de columna)
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
N_COLUMNAS = 6      # cuántas columnas dejan ANOVA y RFE, como en la clase (aquí es la mitad de 12)
MARGEN = 0.01       # el mismo margen del proyecto: lo más completo solo gana si mejora en más de 0.01

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

---
## Preparación: los mismos datos y la misma partición del proyecto

In [ ]:
df = pd.read_csv(DATA_DIR / "heart_disease_uci.csv")

# Las mismas reglas del proyecto: un colesterol o una presión de 0 son faltantes mal capturados
df["chol"] = df["chol"].replace(0, np.nan)
df["trestbps"] = df["trestbps"].replace(0, np.nan)
for col in ["fbs", "exang"]:
    df[col] = df[col].map({True: "sí", False: "no"})
df["target"] = (df["num"] > 0).astype(int)

NUMERICAS = ["age", "trestbps", "chol", "thalch", "oldpeak"]
CATEGORICAS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
COLUMNAS = NUMERICAS + CATEGORICAS
print("Pacientes:", len(df), "| columnas candidatas:", len(COLUMNAS))

In [ ]:
X = df[COLUMNAS]
y = df["target"]

X_resto, X_test, y_resto, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_resto, y_resto, test_size=0.25, stratify=y_resto, random_state=SEED)

display(pd.DataFrame({
    "pacientes": [len(y_train), len(y_val), len(y_test)],
    "% con enfermedad": [round(y_train.mean() * 100, 1), round(y_val.mean() * 100, 1), round(y_test.mean() * 100, 1)],
}, index=["entrenamiento", "validación", "prueba"]))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
f2 = make_scorer(fbeta_score, beta=2)

Quedan los mismos 552 pacientes en entrenamiento, 184 en validación y 184 en prueba que en el proyecto, con 55.3%, 55.4% y 55.4% de enfermos.

### Columnas originales y columnas de 0 y 1

Uso el mismo preprocesamiento del proyecto: la mediana para completar los valores numéricos faltantes, `faltante` como categoría propia y `OneHotEncoder` para las categóricas. `OneHotEncoder` convierte cada categórica en varias columnas de 0 y 1; por ejemplo, `cp` se vuelve `cp_asymptomatic`, `cp_atypical angina` y así.

Como quiero seleccionar las **columnas originales**, si una técnica conserva al menos una de las columnas de 0 y 1 de `cp`, cuento a `cp` como conservada. La función `columna_original` identifica de dónde salió cada columna.

In [ ]:
def armar_preprocesador(escalar, numericas=NUMERICAS, categoricas=CATEGORICAS):
    pasos_num = [("imputador", SimpleImputer(strategy="median"))]
    if escalar:
        pasos_num.append(("escalador", StandardScaler()))
    pasos_cat = [("imputador", SimpleImputer(strategy="constant", fill_value="faltante")),
                 ("codificador", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
    return ColumnTransformer([("num", Pipeline(pasos_num), numericas), ("cat", Pipeline(pasos_cat), categoricas)])


def armar_modelo(nombre, escalar, parametros=None, categoricas=CATEGORICAS, numericas=NUMERICAS):
    modelos = {
        "Regresión logística": LogisticRegression(max_iter=5000, random_state=SEED),
        "KNN": KNeighborsClassifier(),
        "Árbol de decisión": DecisionTreeClassifier(random_state=SEED),
    }
    modelo = modelos[nombre]
    if parametros:
        modelo.set_params(**parametros)
    return Pipeline([("preprocesador", armar_preprocesador(escalar, numericas, categoricas)), ("modelo", modelo)])


ESCALAR = {"Regresión logística": True, "KNN": True, "Árbol de decisión": False}


def metricas(modelo, X_datos, y_datos):
    pred = modelo.predict(X_datos)
    prob = modelo.predict_proba(X_datos)[:, 1]
    return {"accuracy": accuracy_score(y_datos, pred), "recall": recall_score(y_datos, pred), "precision": precision_score(y_datos, pred, zero_division=0),
            "F2": fbeta_score(y_datos, pred, beta=2), "AUC": roc_auc_score(y_datos, prob)}


def separar(columnas):
    # Parte una lista de columnas en numéricas y categóricas, respetando el orden de la tabla
    return [c for c in NUMERICAS if c in columnas], [c for c in CATEGORICAS if c in columnas]


def columna_original(nombre):
    # "num__age" viene de age y "cat__cp_asymptomatic" viene de cp
    tipo, resto = nombre.split("__", 1)
    if tipo == "num":
        return resto
    return next(c for c in CATEGORICAS if resto.startswith(c + "_"))


preprocesador = armar_preprocesador(escalar=True).fit(X_train)
Z_train = preprocesador.transform(X_train)
origen = pd.Series([columna_original(n) for n in Z_train.columns], index=Z_train.columns)

print("Columnas después del preprocesamiento:", Z_train.shape[1])
display(origen.value_counts().rename("columnas de 0 y 1 o numéricas").to_frame().T)

**Qué veo.** Las 12 columnas se vuelven 28: cada numérica sigue siendo una, `cp`, `slope` y `thal` se vuelven 4 cada una, `fbs`, `restecg` y `exang` 3, y `sex` 2.

### Con qué métrica comparo los subconjuntos

En el proyecto elegí con F2, que combina recall y precision y le da el doble de peso al recall. El recall es la proporción de enfermos que el modelo detecta, y la precision es la proporción de enfermos de verdad entre los pacientes que marca como enfermos.

Antes de usar F2 para comparar subconjuntos de columnas, lo comparo contra un modelo que no aprende nada: decirle "enfermo" a todos los pacientes. Uso validación cruzada, que divide el entrenamiento en 5 partes y, por turnos, entrena con 4 y evalúa con la que queda.

In [ ]:
modelo_proyecto = armar_modelo("Regresión logística", True, {"C": 1})
todos_enfermos = DummyClassifier(strategy="constant", constant=1)

filas = []
for nombre, modelo in {"Decirle enfermo a todos": todos_enfermos, "Modelo del proyecto (12 columnas, C = 1)": modelo_proyecto}.items():
    filas.append({"modelo": nombre,
                  "F2 (validación cruzada)": cross_val_score(modelo, X_train, y_train, cv=cv, scoring=f2).mean(),
                  "AUC (validación cruzada)": cross_val_score(modelo, X_train, y_train, cv=cv, scoring="roc_auc").mean()})
display(pd.DataFrame(filas).set_index("modelo").round(4))

**Qué veo.** Decirle enfermo a todos saca un F2 de 0.8606 en validación cruzada, más que el modelo del proyecto (0.8541). La razón es que en entrenamiento el 55.3% está enfermo: si digo enfermo a todos, el recall es 1 y la precision es 0.553, y con el recall contando el doble eso da un F2 de 0.86. Entonces F2 no distingue un subconjunto útil de uno que solo empuja al modelo a decir enfermo a casi todos.

El **AUC** (área bajo la curva ROC) mide qué tan seguido el modelo le da más probabilidad de enfermedad a un enfermo que a un sano. Vale 0.5 si el modelo no distingue nada y 1 si siempre los ordena bien, y no depende del umbral de 0.5. Decirle enfermo a todos saca 0.5 y el modelo del proyecto 0.8817.

**Decisión: comparo los subconjuntos con AUC.** F2, recall y precision los sigo reportando, como en el proyecto. En la sección 4 se ve qué pasa si selecciono con F2.

---
## 1. Filtro de varianza

Como en la clase, `VarianceThreshold(threshold=0)` quita solo las columnas que tienen siempre el mismo valor. No toma en cuenta si el paciente está enfermo. Lo aplico antes de escalar, a las 28 columnas, tanto las numéricas como las de 0 y 1.

In [ ]:
Z_sin_escalar = armar_preprocesador(escalar=False).fit_transform(X_train)
filtro_varianza = VarianceThreshold(threshold=0).fit(Z_sin_escalar)

print("Columnas que quita el filtro:", list(Z_sin_escalar.columns[~filtro_varianza.get_support()]))
conservadas = {columna_original(n) for n in Z_sin_escalar.columns[filtro_varianza.get_support()]}
columnas_varianza = [c for c in COLUMNAS if c in conservadas]
columnas_varianza

**Qué veo.** No quita ninguna: ninguna columna es constante en entrenamiento, así que se quedan las 12.

---
## 2. Filtro de correlación entre características

Calculo la correlación de Pearson sobre las columnas numéricas y las de 0 y 1. Para cada par de columnas originales tomo la correlación absoluta más alta entre las columnas que las representan. Por ejemplo, entre `cp` y `exang` es la más alta de las 12 combinaciones de sus categorías. No comparo entre sí las categorías de una misma columna, porque en estos datos `sex_Male` y `sex_Female` tienen una correlación de -1.

Sigo la regla de la clase: recorro las columnas en su orden y conservo una si su correlación con todas las ya conservadas es menor que 0.85.

In [ ]:
correlacion = Z_train.corr().abs()

correlacion_columnas = pd.DataFrame(np.nan, index=COLUMNAS, columns=COLUMNAS)
for a in COLUMNAS:
    for b in COLUMNAS:
        if a != b:
            correlacion_columnas.loc[a, b] = correlacion.loc[origen == a, origen == b].values.max()

plt.figure(figsize=(9, 7))
sns.heatmap(correlacion_columnas, annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=1)
plt.title("Correlación absoluta más alta entre las categorías de cada par de columnas (entrenamiento)")
plt.tight_layout()
plt.show()

In [ ]:
columnas_correlacion = []
for columna in COLUMNAS:
    if (correlacion_columnas.loc[columna, columnas_correlacion] < 0.85).all():
        columnas_correlacion.append(columna)
print("Se conservan", len(columnas_correlacion), "columnas:", columnas_correlacion)

# Los pares de columnas de 0 y 1 más correlacionados que vienen de columnas originales distintas
pares = correlacion.where(np.triu(np.ones(correlacion.shape, dtype=bool), k=1)).stack()
pares = pares[[origen[a] != origen[b] for a, b in pares.index]].sort_values(ascending=False)
display(pares.head(5).round(3).rename("correlación absoluta").to_frame())

falta_slope, falta_thal = X_train["slope"].isna(), X_train["thal"].isna()
print("Pacientes sin slope:", falta_slope.sum(), "| de ellos, también sin thal:", (falta_slope & falta_thal).sum())

**Qué veo.** La mayor correlación absoluta entre columnas originales distintas es de 0.47, entre `slope_faltante` y `thal_faltante`. Está muy por debajo de 0.85, así que el filtro conserva las 12 columnas.

Los pares con mayor correlación están relacionados con los faltantes: de los 176 pacientes de entrenamiento sin `slope`, a 151 también les falta `thal`. Coincide con lo que vi en el proyecto: hay hospitales que no midieron esas columnas. Entre las numéricas, la mayor correlación absoluta es la de `age` con `thalch` (0.37).

---
## 3. Selección univariada: ANOVA

`f_classif` calcula, para cada columna por separado, qué tanto difiere su promedio entre enfermos y sanos en relación con su variación dentro de cada grupo. Lo aplico a las columnas numéricas y a las de 0 y 1. A cada columna original le asigno la mayor puntuación entre las columnas que la representan y conservo las 6 con mayor puntuación.

In [ ]:
puntuacion_f, _ = f_classif(Z_train, y_train)
anova_categorias = pd.Series(puntuacion_f, index=Z_train.columns)
anova_columnas = anova_categorias.groupby(origen).max().sort_values(ascending=False)

columnas_anova = [c for c in COLUMNAS if c in anova_columnas.index[:N_COLUMNAS]]
display(anova_columnas.round(1).rename("puntuación F").to_frame().T)
columnas_anova

In [ ]:
anova_columnas.sort_values().plot.barh(figsize=(8, 5), color="#8172b3")
plt.axhline(len(COLUMNAS) - N_COLUMNAS - 0.5, color="black", linestyle="--", linewidth=0.8)
plt.title("ANOVA · Puntuación de cada columna en entrenamiento (arriba de la línea: las 6 que se quedan)")
plt.xlabel("Puntuación F de su mejor categoría")
plt.tight_layout()
plt.show()

**Qué veo.** Las 6 que se quedan son `cp` (166.5), `exang` (130.3), `thalch` (101.9), `oldpeak` (81.2), `slope` (73.5) y `age` (71.2). `sex` queda justo afuera con 70.1. Las tres más bajas son `restecg` (9.6), `trestbps` (8.3) y `chol` (5.1): por separado casi no cambian entre enfermos y sanos.

---
## 4. Selección hacia adelante (forward)

Empiezo sin columnas, pruebo agregar cada una, me quedo con la que da el mejor AUC en validación cruzada y repito. Me detengo cuando la mejor incorporación ya no sube el AUC al menos `1e-6`, igual que `SequentialFeatureSelector` con `n_features_to_select="auto"` y `tol=1e-6`. Evalúo cada subconjunto con la regresión logística que elegí en el proyecto (`C = 1`).

No uso `SequentialFeatureSelector` directamente porque marca error con los valores faltantes (`NaN`) de `chol` o `trestbps`. Además, si recibe las columnas ya codificadas, trata las categorías de `cp` como columnas distintas. Por eso escribo el procedimiento con un ciclo que selecciona columnas originales completas, y después comparo su resultado con el de scikit-learn usando solo las numéricas.

In [ ]:
def puntaje(columnas, metrica="roc_auc"):
    numericas, categoricas = separar(columnas)
    modelo = armar_modelo("Regresión logística", True, {"C": 1}, categoricas=categoricas, numericas=numericas)
    return cross_val_score(modelo, X_train[columnas], y_train, cv=cv, scoring=metrica).mean()


def hacia_adelante(candidatas, puntuar, tol=1e-6):
    elegidas, anterior, pasos = [], -np.inf, []
    while len(elegidas) < len(candidatas) - 1:          # con "auto", scikit-learn llega a lo más a una menos que el total
        pruebas = {c: puntuar(elegidas + [c]) for c in candidatas if c not in elegidas}
        mejor = max(pruebas, key=pruebas.get)
        if pruebas[mejor] - anterior < tol:
            break
        elegidas.append(mejor)
        anterior = pruebas[mejor]
        pasos.append({"paso": len(elegidas), "agrega": mejor, "puntaje": anterior})
    return elegidas, pd.DataFrame(pasos).set_index("paso")


def hacia_atras(candidatas, puntuar, tol=0.0):
    quedan, anterior, pasos = list(candidatas), -np.inf, []
    while len(quedan) > 1:
        pruebas = {c: puntuar([k for k in quedan if k != c]) for c in quedan}
        mejor = max(pruebas, key=pruebas.get)
        if pruebas[mejor] - anterior < tol:
            break
        quedan.remove(mejor)
        anterior = pruebas[mejor]
        pasos.append({"paso": len(pasos) + 1, "quita": mejor, "puntaje": anterior})
    return quedan, pd.DataFrame(pasos).set_index("paso")

**Comprobación con scikit-learn.** Con solo las 5 numéricas, ya rellenadas con la mediana de entrenamiento, sí puedo ejecutar `SequentialFeatureSelector`. Si mis funciones están bien, tienen que elegir lo mismo.

In [ ]:
X_num = X_train[NUMERICAS].fillna(X_train[NUMERICAS].median())
lineal_num = Pipeline([("escalador", StandardScaler()), ("modelo", LogisticRegression(max_iter=5000, random_state=SEED))])


def puntaje_numericas(columnas):
    return cross_val_score(lineal_num, X_num[columnas], y_train, cv=cv, scoring="roc_auc").mean()


for direccion, tol, funcion in [("forward", 1e-6, hacia_adelante), ("backward", 0.0, hacia_atras)]:
    sfs = SequentialFeatureSelector(lineal_num, n_features_to_select="auto", tol=tol, direction=direccion,
                                    scoring="roc_auc", cv=cv).fit(X_num, y_train)
    mias, _ = funcion(NUMERICAS, puntaje_numericas, tol)
    print(f"{direccion}: scikit-learn {list(X_num.columns[sfs.get_support()])} | mi función {[c for c in NUMERICAS if c in mias]}")

Las selecciones coinciden en las dos direcciones. Ahora aplico mis funciones a las 12 columnas originales.

In [ ]:
elegidas_forward, pasos_forward = hacia_adelante(COLUMNAS, puntaje)
columnas_forward = [c for c in COLUMNAS if c in elegidas_forward]
display(pasos_forward.round(4))
print("Quedan fuera:", [c for c in COLUMNAS if c not in elegidas_forward])

Y para ver por qué no uso F2, repito lo mismo con F2 como puntaje:

In [ ]:
elegidas_forward_f2, pasos_forward_f2 = hacia_adelante(COLUMNAS, lambda columnas: puntaje(columnas, f2))
display(pasos_forward_f2.round(4))

**Qué veo.** Con AUC, forward agrega primero `cp` (0.7616), luego `oldpeak` (0.8131), `sex` (0.8397), `thalch` (0.8554), `slope` (0.8638), `fbs` (0.8754), `age` (0.8820), `thal` (0.8843), `exang` (0.8850) y `chol` (0.8863), y ahí se detiene. Se queda con 10 columnas y deja fuera `trestbps` y `restecg`.

Con F2 se detiene con solo dos columnas, `sex` (0.8341) y `fbs` (0.8387), menos que el 0.8606 de decirle enfermo a todos. Esto respalda que use AUC para comparar los subconjuntos.

---
## 5. Selección hacia atrás (backward)

Parto de las 12 columnas y en cada paso quito la que, al salir, deja el mejor AUC. Con `tol=0.0` sigo mientras el AUC se mantenga o suba respecto al paso anterior. Como en `SequentialFeatureSelector`, la primera eliminación siempre se hace. A partir de ahí comparo cada paso con el anterior, y no con el modelo de las 12 columnas.

In [ ]:
columnas_backward, pasos_backward = hacia_atras(COLUMNAS, puntaje)
display(pasos_backward.round(4))
print("AUC con las 12 columnas:", round(puntaje(COLUMNAS), 4))
print("Se quedan", len(columnas_backward), "columnas:", columnas_backward)

In [ ]:
print("En ambos:", [c for c in COLUMNAS if c in columnas_forward and c in columnas_backward])
print("Solo forward:", [c for c in COLUMNAS if c in columnas_forward and c not in columnas_backward])
print("Solo backward:", [c for c in COLUMNAS if c in columnas_backward and c not in columnas_forward])

**Qué veo.** Backward quita `trestbps` (0.8845), luego `restecg` (0.8863) y luego `thal` (0.8865). Después, quitar cualquier otra columna baja el AUC, así que se queda con 9 columnas. En esta validación cruzada, las 9 sacan más AUC que las 12 (0.8817).

Forward y backward coinciden en sacar `trestbps` y `restecg`. Solo difieren en `thal`: forward la agrega en el paso 8 y backward la quita. Cada método agrega o quita columnas sin volver a revisar lo que ya decidió, así que pueden terminar con subconjuntos distintos.

---
## 6. Eliminación recursiva: RFE

RFE entrena el modelo, quita la columna con menos peso y vuelve a entrenar hasta dejar 6. A diferencia de backward, no evalúa cada eliminación con validación cruzada: usa los coeficientes del modelo ya entrenado. Uso la regresión logística del proyecto, que sí tiene coeficientes.

Como en la clase, el peso de cada columna es su coeficiente al cuadrado. Para una categórica sumo los cuadrados de los coeficientes de sus categorías, así que el peso de `cp` es la suma de los cuadrados de sus 4 coeficientes.

In [ ]:
def rfe_por_columnas(candidatas, n_final):
    quedan, pasos = list(candidatas), []
    while len(quedan) > n_final:
        numericas, categoricas = separar(quedan)
        modelo = armar_modelo("Regresión logística", True, {"C": 1}, categoricas=categoricas, numericas=numericas)
        modelo.fit(X_train[quedan], y_train)
        nombres = modelo.named_steps["preprocesador"].get_feature_names_out()
        coeficientes = pd.Series(modelo.named_steps["modelo"].coef_[0], index=nombres)
        peso = (coeficientes ** 2).groupby([columna_original(n) for n in nombres]).sum()
        pasos.append({"paso": len(pasos) + 1, "quita": peso.idxmin(), "peso": peso.min()})
        quedan.remove(peso.idxmin())
    return quedan, pd.DataFrame(pasos).set_index("paso")


columnas_rfe, pasos_rfe = rfe_por_columnas(COLUMNAS, N_COLUMNAS)
display(pasos_rfe.round(4))
columnas_rfe

In [ ]:
# Comprobación: con solo las numéricas, mi función y RFE de scikit-learn deben quitar lo mismo
rfe_sklearn = RFE(LogisticRegression(max_iter=5000, random_state=SEED), n_features_to_select=2)
rfe_sklearn.fit(StandardScaler().fit_transform(X_num), y_train)
mias, _ = rfe_por_columnas(NUMERICAS, 2)
print("scikit-learn:", list(X_num.columns[rfe_sklearn.get_support()]), "| mi función:", mias)

**Qué veo.** Con solo las numéricas, mi función y la de scikit-learn se quedan con las mismas dos (`thalch` y `oldpeak`). Con las 12, RFE quita `trestbps`, `restecg`, `chol`, `thalch`, `oldpeak` y `age`, y se queda con las 6 categóricas que no son `restecg`: `sex`, `cp`, `fbs`, `exang`, `slope` y `thal`.

La forma en que calculo el peso favorece a las categóricas: cada una suma varios coeficientes al cuadrado (`cp` suma 4), mientras que una numérica solo aporta uno. Además, las numéricas están estandarizadas y las de 0 y 1 no, así que sus coeficientes no están en la misma escala. Como las otras técnicas sí conservan `oldpeak` y `thalch`, tomo este resultado con cuidado.

---
## 7. Selección con Lasso

Lasso es una regresión lineal que, según qué tanto se le penalice (`alpha`), puede llevar algunos coeficientes a exactamente cero. Como el target ya es 0 o 1, ajusto una sola regresión sobre las 28 columnas estandarizadas. Uso `alpha=0.05`, como en la clase, y conservo una columna si al menos una de sus categorías queda con coeficiente distinto de cero.

In [ ]:
lasso = Lasso(alpha=0.05).fit(Z_train, y_train)
pesos_lasso = pd.Series(lasso.coef_, index=Z_train.columns)

conservar = (pesos_lasso != 0).groupby(origen).any()
columnas_lasso = [c for c in COLUMNAS if conservar[c]]
display(pesos_lasso[pesos_lasso != 0].round(3).rename("coeficiente").to_frame().T)
columnas_lasso

Tomo `alpha=0.05` de la clase, no lo busqué yo. Para revisar cuánto depende la selección de ese valor, repito con otros:

In [ ]:
filas = []
for alpha in [0.01, 0.02, 0.05, 0.1]:
    pesos = pd.Series(Lasso(alpha=alpha).fit(Z_train, y_train).coef_, index=Z_train.columns)
    quedan = [c for c in COLUMNAS if (pesos != 0).groupby(origen).any()[c]]
    filas.append({"alpha": alpha, "columnas que se quedan": len(quedan), "cuáles": ", ".join(quedan)})
display(pd.DataFrame(filas).set_index("alpha"))

**Qué veo.** Con `alpha=0.05` se quedan 5 columnas: `age`, `thalch`, `oldpeak`, `cp` y `exang`. El resultado depende mucho de `alpha`: con 0.01 se quedan 11, con 0.02 se quedan 10 y con 0.1 solo 3 (`age`, `thalch` y `oldpeak`). Lo que no cambia es que `trestbps` sale en los cuatro casos.

---
## 8. Extensión: búsqueda exhaustiva

Las 12 columnas tienen 2¹² - 1 = 4095 subconjuntos no vacíos. Como evaluar cada uno requiere 5 entrenamientos, primero mido cuánto tarda uno.

In [ ]:
inicio = time.perf_counter()
puntaje(COLUMNAS)
segundos = time.perf_counter() - inicio
print(f"Una evaluación tarda {segundos:.2f} segundos; los 4095 subconjuntos tardarían unos {segundos * 4095 / 60:.0f} minutos")

La búsqueda completa tarda demasiado para repetirla cada vez que corro el notebook. Como en la clase, defino las candidatas antes de la búsqueda y pruebo todos sus subconjuntos. Las elijo usando solo entrenamiento: las 7 con mayor puntuación de ANOVA. Son 2⁷ - 1 = 127 subconjuntos. Si dos empatan, prefiero el que tiene menos columnas.

In [ ]:
candidatas_exhaustiva = anova_columnas.index[:7].tolist()
print("Candidatas:", candidatas_exhaustiva)

resultados_exhaustiva = []
for tamano in range(1, len(candidatas_exhaustiva) + 1):
    for subconjunto in combinations(candidatas_exhaustiva, tamano):
        columnas = [c for c in COLUMNAS if c in subconjunto]
        resultados_exhaustiva.append({"columnas": columnas, "cuántas": len(columnas), "AUC": puntaje(columnas)})

tabla_exhaustiva = pd.DataFrame(resultados_exhaustiva).sort_values(["AUC", "cuántas"], ascending=[False, True])
display(tabla_exhaustiva.head().round(4))
columnas_exhaustiva = tabla_exhaustiva.iloc[0]["columnas"]
columnas_exhaustiva

**Qué veo.** Entre las 7 candidatas, el mayor AUC se obtiene usando las 7 (0.8755). Quitar `thalch` deja 0.8740 y quitar `oldpeak`, 0.8728. Esta búsqueda es exhaustiva solo entre esas 7 columnas y para la regresión logística con `C = 1`, así que su alcance es distinto al de forward y backward, que revisan las 12.

---
## 9. ¿Qué columnas eligió cada técnica?

En la gráfica, cada fila es una columna original de los datos y cada columna es una técnica. El 1 indica que la técnica la conservó.

In [ ]:
conjuntos = {
    "Sin selección": COLUMNAS,
    "Varianza cero": columnas_varianza,
    "Correlación": columnas_correlacion,
    "ANOVA": columnas_anova,
    "Hacia adelante": columnas_forward,
    "Hacia atrás": columnas_backward,
    "RFE": columnas_rfe,
    "Lasso": columnas_lasso,
    "Exhaustiva (7 candidatas)": columnas_exhaustiva,
}

matriz_columnas = pd.DataFrame({tecnica: [int(c in columnas) for c in COLUMNAS] for tecnica, columnas in conjuntos.items()}, index=COLUMNAS)
matriz_columnas["técnicas que miran el target"] = matriz_columnas[["ANOVA", "Hacia adelante", "Hacia atrás", "RFE", "Lasso", "Exhaustiva (7 candidatas)"]].sum(axis=1)

plt.figure(figsize=(12, 6))
sns.heatmap(matriz_columnas.drop(columns="técnicas que miran el target"), annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Columnas seleccionadas · 1: conservada, 0: descartada")
plt.xlabel("Técnica de selección")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Columna")
plt.tight_layout()
plt.show()

display(matriz_columnas["técnicas que miran el target"].sort_values(ascending=False).to_frame().T)

**Qué veo.** En este reparto, varianza y correlación no quitan nada, así que dan lo mismo que no seleccionar. De las seis técnicas que miran el target:

- `cp` y `exang` las conservan las seis.
- `age`, `thalch`, `oldpeak` y `slope` las conservan cinco, y `sex` cuatro.
- `trestbps` (presión en reposo) y `restecg` (electrocardiograma en reposo) no las conserva ninguna. Es el resultado más claro: casi no ayudan cuando ya están las demás columnas.
- `fbs` la conservan forward, backward y RFE. En el proyecto vi que su categoría `faltante` está relacionada con el hospital de Suiza, así que puede ser que la conserven por esa relación con el hospital y no por la glucosa.

---
## 10. Entrenar los tres modelos con cada subconjunto

Para cada subconjunto busco los hiperparámetros de los tres modelos, es decir, los valores que se fijan antes de entrenar (como `C` o el número de vecinos). Uso `GridSearchCV` con las mismas cuadrículas del proyecto, la misma validación cruzada `cv` de 5 partes y AUC como métrica. Después mido cada modelo en validación.

Seleccioné las columnas usando todo el entrenamiento, así que la validación cruzada de aquí puede salir optimista. Por eso elijo el modelo con los resultados del conjunto de validación.

In [ ]:
cuadriculas = {
    "Regresión logística": {"modelo__C": [0.01, 0.1, 1, 10, 100], "modelo__class_weight": [None, "balanced"]},
    "KNN": {"modelo__n_neighbors": list(range(1, 32, 2)), "modelo__weights": ["uniform", "distance"], "modelo__p": [1, 2]},
    "Árbol de decisión": {"modelo__max_depth": [2, 3, 4, 5, 6, 8], "modelo__min_samples_leaf": [1, 5, 10, 20],
                          "modelo__criterion": ["gini", "entropy"], "modelo__class_weight": [None, "balanced"]},
}

busquedas = {}
filas = []
for tecnica, columnas in conjuntos.items():
    numericas, categoricas = separar(columnas)
    for nombre, cuadricula in cuadriculas.items():
        clave = (tuple(columnas), nombre)
        if clave not in busquedas:      # varianza y correlación dejan las 12 columnas: no repito esa búsqueda
            busqueda = GridSearchCV(armar_modelo(nombre, ESCALAR[nombre], categoricas=categoricas, numericas=numericas),
                                    cuadricula, cv=cv, scoring="roc_auc")
            busquedas[clave] = busqueda.fit(X_train[columnas], y_train)
        busqueda = busquedas[clave]
        filas.append({"técnica": tecnica, "columnas": len(columnas), "modelo": nombre, "AUC validación cruzada": busqueda.best_score_,
                      **{f"{k} validación": v for k, v in metricas(busqueda, X_val[columnas], y_val).items()},
                      "parámetros": {k.replace("modelo__", ""): v for k, v in busqueda.best_params_.items()}})
resultados = pd.DataFrame(filas)

In [ ]:
orden_modelos = ["Regresión logística", "KNN", "Árbol de decisión"]
for valor in ["AUC validación cruzada", "AUC validación"]:
    tabla = resultados.pivot(index=["técnica", "columnas"], columns="modelo", values=valor)
    tabla = tabla.reindex(conjuntos.keys(), level="técnica")[orden_modelos]
    display(tabla.round(4).rename_axis(columns=valor))

**Qué veo.** En validación cruzada, forward con KNN es el más alto (0.8911), seguido de forward con regresión logística (0.8897) y backward con regresión logística (0.8875). Con las 12 columnas, la regresión logística saca 0.8848 y KNN 0.8885.

En validación, el mejor es forward con KNN (0.8748). Le siguen las 12 columnas con KNN (0.8692), las 12 con regresión logística (0.8687), forward con regresión logística (0.8678) y backward con regresión logística (0.8651). ANOVA y Lasso, con 6 y 5 columnas, se quedan entre 0.81 y 0.86. El árbol queda último con cualquier subconjunto, como en el proyecto.

## Elección y evaluación final

Regla, escrita antes de ver la prueba: tomo el mayor AUC en validación y, entre los que quedan a menos de 0.01 de él (el `MARGEN` del proyecto), me quedo con el que tiene menos columnas. Si empatan en columnas, gana el de mayor AUC. Después uso la prueba una sola vez.

In [ ]:
mejor_auc = resultados["AUC validación"].max()
cercanos = resultados[resultados["AUC validación"] >= mejor_auc - MARGEN]
display(cercanos[["técnica", "columnas", "modelo", "AUC validación", "F2 validación", "recall validación"]].round(4))

elegido = cercanos.sort_values(["columnas", "AUC validación"], ascending=[True, False], kind="stable").iloc[0]
tecnica_elegida, modelo_elegido = elegido["técnica"], elegido["modelo"]
columnas_elegidas = conjuntos[tecnica_elegida]
print("Elegido:", tecnica_elegida, "·", modelo_elegido, "·", elegido["parámetros"])
print("Columnas:", columnas_elegidas)

Con la regla elijo **backward con regresión logística**: 9 columnas y `C = 0.1`. Su AUC en validación (0.8651) queda dentro del margen por muy poco, a 0.0097 del mejor. Ahora lo evalúo en prueba.

Para compararlo, muestro también el mismo modelo con las 12 columnas, el modelo del proyecto y decirle enfermo a todos. Esas tres filas son solo de comparación: la elección ya está hecha.

In [ ]:
final = busquedas[(tuple(columnas_elegidas), modelo_elegido)]
mismo_con_12 = busquedas[(tuple(COLUMNAS), modelo_elegido)]
modelo_proyecto.fit(X_train, y_train)
todos_enfermos.fit(X_train, y_train)

comparacion = {
    f"Elegido: {tecnica_elegida} + {modelo_elegido} ({len(columnas_elegidas)} columnas)": (final, columnas_elegidas),
    f"{modelo_elegido} con las 12 columnas": (mismo_con_12, COLUMNAS),
    "Modelo del proyecto (12 columnas, C = 1)": (modelo_proyecto, COLUMNAS),
    "Decirle enfermo a todos": (todos_enfermos, COLUMNAS),
}

filas = []
for nombre, (modelo, columnas) in comparacion.items():
    tn, fp, fn, tp = confusion_matrix(y_test, modelo.predict(X_test[columnas])).ravel()
    filas.append({"modelo": nombre, **metricas(modelo, X_test[columnas], y_test),
                  "enfermos que se dicen sanos": int(fn), "sanos que se dicen enfermos": int(fp)})
display(pd.DataFrame(filas).set_index("modelo").round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, nombre in zip(axes, list(comparacion)[:3:2]):
    modelo, columnas = comparacion[nombre]
    matriz = confusion_matrix(y_test, modelo.predict(X_test[columnas]))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["sin enfermedad", "con enfermedad"], yticklabels=["sin enfermedad", "con enfermedad"])
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

**Qué veo.** En prueba (184 pacientes, 102 enfermos), el modelo elegido saca un AUC de 0.9016, F2 de 0.8591, recall de 0.8725 y precision de 0.8091. Deja pasar a 13 enfermos y marca como enfermos a 21 sanos.

Frente al mismo modelo con las 12 columnas (AUC de 0.9057, 12 enfermos sin detectar y 21 sanos mal marcados), quitar `trestbps`, `restecg` y `thal` reduce el AUC en 0.004 y deja sin detectar a un enfermo más. El modelo del proyecto sale mejor en prueba: AUC de 0.9077, 9 enfermos sin detectar y 18 sanos mal marcados.

La diferencia entre el modelo del proyecto y el elegido es de 4 enfermos sin detectar de 102: 3 aparecen al cambiar `C` (0.1, elegido con AUC, contra 1, elegido con F2 en el proyecto) y 1 al quitar las tres columnas. La diferencia de F2 (0.8960 contra 0.8591) es menor que lo que el F2 de prueba se movió entre repartos en el proyecto (de 0.80 a 0.91). Aun así, el resultado no confirma que quitar columnas mejore el modelo.

Decirle enfermo a todos saca un F2 de 0.8615 en prueba, casi lo mismo que el elegido, y un AUC de 0.5. Es otra muestra de que F2 solo no basta para comparar.

### Un hospital que el modelo no vio

En el proyecto, el modelo bajaba bastante cuando lo probaba con un hospital que no estaba en el entrenamiento. Repito esa prueba con el modelo elegido y sus hiperparámetros, una vez con las 12 columnas y otra con las 9. Esto no cambia la elección; sirve para ver si quitar columnas ayuda con un hospital nuevo.

In [ ]:
parametros_elegidos = {k.replace("modelo__", ""): v for k, v in final.best_params_.items()}
filas = []
for fuente in df["dataset"].unique():
    entrena = df["dataset"] != fuente
    fila = {"hospital que se deja fuera": fuente, "pacientes": int((~entrena).sum())}
    for etiqueta, columnas in {"12 columnas": COLUMNAS, f"{len(columnas_elegidas)} columnas": columnas_elegidas}.items():
        numericas, categoricas = separar(columnas)
        modelo = armar_modelo(modelo_elegido, ESCALAR[modelo_elegido], parametros_elegidos, categoricas=categoricas, numericas=numericas)
        modelo.fit(df.loc[entrena, columnas], df.loc[entrena, "target"])
        fila[f"AUC con {etiqueta}"] = roc_auc_score(df.loc[~entrena, "target"], modelo.predict_proba(df.loc[~entrena, columnas])[:, 1])
    filas.append(fila)

hospitales = pd.DataFrame(filas).set_index("hospital que se deja fuera")
display(hospitales.round(3))
print("Promedio de los cuatro hospitales:", hospitales.drop(columns="pacientes").mean().round(3).to_dict())

**Qué veo.** Con un hospital nuevo, las 9 columnas y las 12 dan resultados cercanos: en promedio, 0.818 contra 0.813 de AUC. Con las 9 sube un poco en Suiza (de 0.728 a 0.743) y en VA Long Beach (de 0.745 a 0.755), y baja un poco en Cleveland (de 0.879 a 0.875) y en Hungría (de 0.898 a 0.897). Quitar columnas no arregla el problema del origen de los datos y tampoco lo empeora.

---
## Conclusión

**Puedo pasar de las 12 columnas a 9** (sin `trestbps`, `restecg` ni `thal`): la regresión logística queda a menos de 0.01 del mayor AUC en validación y pierde 0.004 de AUC en prueba frente al mismo modelo con las 12 columnas. Pero quitar columnas no mejora el modelo: el del proyecto, con las 12, sigue dejando pasar menos enfermos en prueba (9 contra 13).

Lo más sólido es lo que se repite entre técnicas:

- `cp` (tipo de dolor de pecho) y `exang` (angina con ejercicio) las conservan las seis técnicas que miran el target.
- `trestbps` y `restecg` no las conserva ninguna.
- Cada técnica que mira el target da un subconjunto distinto, de 5 a 10 columnas. El resultado depende del método y de sus parámetros (`alpha` en Lasso, el número de columnas en ANOVA y RFE).

## Decisiones

| # | Decisión | Opciones | Elección | Evidencia (solo entrenamiento y validación) |
|---|---|---|---|---|
| 1 | Datos, limpieza y partición | Rehacerlos o seguir el proyecto | Seguir el proyecto | Mismos 552, 184 y 184 pacientes |
| 2 | Qué se selecciona | Columnas de 0 y 1 o columnas originales | Columnas originales | Una categoría suelta de `cp` no tiene sentido sin las demás |
| 3 | Métrica para seleccionar | F2 o AUC | AUC | Decirle enfermo a todos saca 0.8606 de F2 y 0.5 de AUC |
| 4 | Forward y backward | `SequentialFeatureSelector` o un ciclo propio | Ciclo propio | El de scikit-learn marca error con los faltantes; con solo numéricas, los dos eligen lo mismo |
| 5 | Candidatas de la exhaustiva | Las 12 o unas cuantas | Las 7 mejores de ANOVA | 4095 subconjuntos tardan demasiado |
| 6 | Modelo final | 27 combinaciones de técnica y modelo | Backward + regresión logística, `C = 0.1` | A menos de 0.01 del mejor en validación y con menos columnas |

## Limitaciones

- **Un solo reparto.** Apliqué las técnicas sobre los mismos 552 pacientes. Con otro reparto podrían elegir otras columnas, y no lo repetí como hice en el proyecto.
- **Mis adaptaciones.** Agrupar las categorías de una columna (la mejor puntuación en ANOVA, la suma de cuadrados en RFE, la correlación más alta) es una decisión mía. Mi forma de calcular el peso en RFE favorece a las categóricas.
- **El modelo elegido entró al margen por 0.0003.** Con un margen de 0.005 la regla habría escogido forward con KNN (10 columnas).
- **`fbs` sigue adentro.** La conservan forward, backward y RFE, y su categoría `faltante` es la pista del hospital que encontré en el proyecto.
- **No ajusté el umbral.** Clasifico como enfermo a quien tiene una probabilidad mayor a 0.5. Con otro umbral podría detectar más enfermos a cambio de marcar como enfermos a más sanos.